# Quantum Graph Attention Networks: From Attention to QGAT

**Author:** <span style="color: #2edee4;">**Niloy Kumar Mondal**</span>

> Senior Undergrad Student  
> Department of Computer Science and Engineering  
> Bangladesh University of Engineering and Technology (BUET)  
> Dhaka, Bangladesh  
> Email: nkm2105044@gmail.com

---

## Abstract

Modern machine learning models do not simply read words, nodes, or edges one by one. They learn **where to look**. Attention helps a model decide which parts of an input matter most, Graph Neural Networks extend learning to connected data, and Graph Attention Networks combine both ideas by letting each node focus on its most important neighbors.

This challenge starts gently with word embeddings and attention, then moves into multi-head attention, GNN message passing, classical GAT/GATv2, and finally the Quantum Graph Attention Network (QGAT) proposed in the paper [Quantum Graph Attention Network: A Novel Quantum Multi-Head Attention Mechanism for Graph Learning](https://arxiv.org/pdf/2508.17630). The goal is not to memorize formulas. The goal is to build intuition, complete small coding tasks, and see how classical attention can be redesigned with quantum data encoding and quantum circuit measurements.

<div style="background-color: #030303; color: #e6e6e6; padding: 18px; border-left: 4px solid #2edee4; border-radius: 5px;">
<strong>Challenge path:</strong> attention in language -> multi-head attention -> graph message passing -> graph attention -> quantum graph attention.
</div>

---

## Table of Contents

0. [Setup and Dataset Links](#setup-and-dataset-links)
1. [Task 1: Attention Is All You Need](#task-1-attention-is-all-you-need)
2. [Task 2: Multi-Head Attention](#task-2-multi-head-attention)
3. [Task 3: Graph Neural Networks](#task-3-graph-neural-networks)
4. [Task 4: Graph Attention Networks](#task-4-graph-attention-networks)
5. [Task 5: Quantum Data Encoding in QGAT](#task-5-quantum-data-encoding-in-qgat)
6. [Task 6: Quantum Graph Attention](#task-6-quantum-graph-attention)
7. [Task 7: New QGAT Architecture Design and Benchmarks](#task-7-new-qgat-architecture-design-and-benchmarks)

---

> **Note:** This notebook includes conceptual questions and coding exercises with `# TODO` sections. Fill the missing parts, then run the grader cell for each task.


## Setup and Dataset Links

Run the installation cell below if your environment is missing any required packages. The graders are provided locally in `grader.py`.


In [ ]:
# %pip install -r requirements.txt


## Importing Grader

Run this cell once before starting the tasks. The grader is provided as a compiled file.


In [ ]:
import importlib.util, sys

spec = importlib.util.spec_from_file_location("grader", "grader.pyc")
grader = importlib.util.module_from_spec(spec)
sys.modules["grader"] = grader
spec.loader.exec_module(grader)


### Optional Benchmark Datasets

The graded tasks in this notebook use small toy examples so that the ideas stay beginner-friendly. If you want to explore the real benchmark datasets mentioned in the QGAT paper, use these official loaders and links:

| Dataset | Task type | Loader / source |
|---|---|---|
| PubMed | Transductive node classification | [PyTorch Geometric Planetoid](https://pytorch-geometric.readthedocs.io/en/stable/generated/torch_geometric.datasets.Planetoid.html) |
| PPI | Inductive node classification | [PyTorch Geometric PPI](https://pytorch-geometric.readthedocs.io/en/latest/generated/torch_geometric.datasets.PPI.html) |
| ogbn-arxiv | Node property prediction | [Open Graph Benchmark](https://ogb.stanford.edu/docs/home/) |
| ogbn-products | Node property prediction | [Open Graph Benchmark](https://ogb.stanford.edu/docs/home/) |
| ogbn-proteins | Node property prediction | [Open Graph Benchmark](https://ogb.stanford.edu/docs/home/) |
| ogbl-collab | Link prediction | [OGB link prediction datasets](https://ogb.stanford.edu/docs/linkprop/) |
| ogbl-citation2 | Link prediction | [OGB link prediction datasets](https://ogb.stanford.edu/docs/linkprop/) |

These datasets can be large, so download them only when you want to run full experiments beyond the graded challenge.


In [ ]:
# Optional dataset packages for real benchmark experiments.
# PyTorch installation can depend on your CPU/GPU setup, so check https://pytorch.org/get-started/ if this fails.
# %pip install torch torch-geometric ogb


In [ ]:
# Optional dataset loading examples.
# Uncomment only after installing torch, torch-geometric, and ogb.

# DATA_ROOT = "./datasets"

# from torch_geometric.datasets import Planetoid, PPI
# pubmed = Planetoid(root=f"{DATA_ROOT}/Planetoid", name="PubMed")
# ppi_train = PPI(root=f"{DATA_ROOT}/PPI", split="train")

# from ogb.nodeproppred import PygNodePropPredDataset
# ogbn_arxiv = PygNodePropPredDataset(name="ogbn-arxiv", root=DATA_ROOT)
# ogbn_products = PygNodePropPredDataset(name="ogbn-products", root=DATA_ROOT)
# ogbn_proteins = PygNodePropPredDataset(name="ogbn-proteins", root=DATA_ROOT)

# from ogb.linkproppred import PygLinkPropPredDataset
# ogbl_collab = PygLinkPropPredDataset(name="ogbl-collab", root=DATA_ROOT)
# ogbl_citation2 = PygLinkPropPredDataset(name="ogbl-citation2", root=DATA_ROOT)


## Task 1: Attention Is All You Need

<div style="background-color: #030303; color: #e6e6e6; padding: 16px; border-left: 4px solid #2edee4; border-radius: 5px;">
<strong>Goal:</strong> Build intuition for embeddings and attention by filling a small attention-percentage table.
</div>

Words are not numbers, so before a model can read a sentence, it changes each word into an **embedding**, which is just a small vector of numbers.  
Words with related meanings or similar roles usually get vectors that are easier to compare.  
**Attention** is the model's way of asking, "For this word, which other words should I focus on?"  
In the sentence **I love quantum computing**, the word **love** should listen carefully to **quantum** and **computing** because they explain what is being loved.  
The final attention percentage tells us how strongly the row word listens to each column word.

Now watch this video carefully to understand the idea more clearly:

https://youtu.be/vkhPtpUiLd8?si=0J3XuVj1JuY1-r5m

### Question

For the sentence **I love quantum computing**, fill the blank table with **attention percentages only**.

Each blank means: **how much attention the row word gives to the column word**. For example, the box at row **love** and column **quantum** asks: *what percentage of love's attention goes to quantum?*

Use the embeddings below and the attention idea from the video to complete the table. Write **final percentages only**, not dot products, scores, or vectors. Each row should add up to **100%**.


Given embeddings:

| Word | Embedding |
|---|---:|
| I | $[1, 0, 0]$ |
| love | $[0, 1, 0]$ |
| quantum | $[0, 1, 1]$ |
| computing | $[0, 1, 2]$ |

---

| Query word \\ Listens to | I | love | quantum | computing |
|---|---:|---:|---:|---:|
| I |  |  |  |  |
| love |  |  |  |  |
| quantum |  |  |  |  |
| computing |  |  |  |  |


In [ ]:
# TODO: Replace the 0.0 values with your calculated attention percentages.
# In every row, write attention from the row word to these words:
# [I, love, quantum, computing]
# Example: the row for love means how much love attends to I, love, quantum, and computing.
attention = [
    [0.0, 0.0, 0.0, 0.0],  # attention from I
    [0.0, 0.0, 0.0, 0.0],  # attention from love
    [0.0, 0.0, 0.0, 0.0],  # attention from quantum
    [0.0, 0.0, 0.0, 0.0],  # attention from computing
]

from grader import grade_attention

if grade_attention(attention):
    # TODO: Run this cell after filling the table correctly to see your heatmap.
    import matplotlib.pyplot as plt

    words = ["I", "love", "quantum", "computing"]

    fig, ax = plt.subplots(figsize=(7, 5))
    heatmap = ax.imshow(attention, cmap="YlGnBu", vmin=0, vmax=100)

    ax.set_title("Attention Heatmap")
    ax.set_xlabel("Listens to")
    ax.set_ylabel("Query word")
    ax.set_xticks(range(len(words)))
    ax.set_yticks(range(len(words)))
    ax.set_xticklabels(words)
    ax.set_yticklabels(words)

    for row_index, row in enumerate(attention):
        for column_index, value in enumerate(row):
            text_color = "white" if value >= 50 else "black"
            ax.text(
                column_index,
                row_index,
                f"{value:.1f}%",
                ha="center",
                va="center",
                color=text_color,
            )

    fig.colorbar(heatmap, ax=ax, label="Attention %")
    plt.tight_layout()
    plt.show()


## Task 2: Multi-Head Attention

<div style="background-color: #030303; color: #e6e6e6; padding: 16px; border-left: 4px solid #2edee4; border-radius: 5px;">
<strong>Goal:</strong> Understand why several attention heads can look at the same sentence in different useful ways.
</div>

Single-head attention gives one attention table for the sentence.  
Multi-head attention is like asking several careful readers to read the same sentence at the same time.  
One head may focus on meaning, another may focus on word position, and another may focus on grammar.  
Each head creates its own attention pattern, so the model gets more than one view of the sentence.  
After that, the Transformer combines the heads to build a richer understanding.

Now watch this video carefully to understand multi-head attention more clearly:

https://youtu.be/42L1q1Z4Ojc?si=aIQzBxork-3tLBki

### Question

For the sentence **I love quantum computing**, imagine that a Transformer uses three attention heads. Fill up the blank boxes with what each head might focus on. There is **no single fixed answer** here; the goal is to show three different reasonable ways the heads might look at the sentence. Use short beginner-friendly phrases such as **meaning**, **nearby words**, **grammar**, **the object being loved**, or **the whole phrase**.

| Attention head | What might this head focus on? | For the word **love**, which word or phrase might it listen to most? |
|---|---|---|
| Head 1 |  |  |
| Head 2 |  |  |
| Head 3 |  |  |


In [ ]:
# TODO: Fill each head with your own short explanation.
# There is no single fixed answer for this conceptual task.
# Each row is: [what this head focuses on, what love listens to most]
# Try to make the three heads focus on different things.
multi_head_answers = [
    ["", ""],  # Head 1
    ["", ""],  # Head 2
    ["", ""],  # Head 3
]

from grader import grade_multi_head_attention

grade_multi_head_attention(multi_head_answers)


## Task 3: Graph Neural Networks

<div style="background-color: #030303; color: #e6e6e6; padding: 16px; border-left: 4px solid #2edee4; border-radius: 5px;">
<strong>Goal:</strong> Trace how information moves through a graph after one and two message-passing layers.
</div>

Some data is not just a list or a sentence; it is a **graph** made of nodes and connections.  
A **Graph Neural Network**, or **GNN**, is a neural network that learns from this kind of connected data.  
Each node starts with its own information, also called node features.  
During message passing, every node collects information from its neighbors and updates itself.  
After one GNN layer, a node knows about its direct neighbors; after two layers, information can travel from neighbors of neighbors.

Imagine this friendship graph:

$$\text{Alice} - \text{Bob} - \text{Carol} - \text{Dan}$$

Edges:

- Alice is connected to Bob.
- Bob is connected to Carol.
- Carol is connected to Dan.

### Question

Fill the conceptual message-passing answers below.

| Node | Who sends messages to this node after 1 GNN layer? | What new node information can reach this node after 2 GNN layers? |
|---|---|---|
| Alice |  |  |
| Bob |  |  |
| Carol |  |  |
| Dan |  |  |


In [ ]:
# TODO: Fill the graph message-passing answers.
# Order does not matter inside each list.
one_layer_neighbors = {
    "Alice": [],
    "Bob": [],
    "Carol": [],
    "Dan": [],
}

# TODO: Fill the new information that can arrive after two GNN layers.
# Do not include direct neighbors here; include only the new two-hop node.
two_layer_new_info = {
    "Alice": [],
    "Bob": [],
    "Carol": [],
    "Dan": [],
}

from grader import grade_gnn_task

grade_gnn_task(one_layer_neighbors, two_layer_new_info)


## Task 4: Graph Attention Networks

<div style="background-color: #030303; color: #e6e6e6; padding: 16px; border-left: 4px solid #2edee4; border-radius: 5px;">
<strong>Goal:</strong> Convert raw neighbor scores into attention percentages, just like the normalization step in GAT.
</div>

A normal GNN lets a node collect messages from its neighbors.  
A **Graph Attention Network**, or **GAT**, makes this smarter by asking: *which neighbor should I listen to more?*  
For a node $i$ and its neighbor $j$, GAT computes an attention weight $\alpha_{ij}$, meaning how important node $j$ is for updating node $i$.  
The feature vector $h_i$ represents node $i$, the matrix $W$ transforms node features, and concatenation $[Wh_i \| Wh_j]$ puts the two node representations side by side.  
Then GAT turns each neighbor's score into a percentage-like weight using LeakyReLU and softmax over all neighbors of node $i$.

**GATv2** keeps the same main idea, but changes the order of operations. Instead of transforming the two nodes separately before scoring them, GATv2 first looks at the source and target node together, then applies the transformation and nonlinearity. This makes the attention score more flexible, because the importance of a neighbor can depend more directly on the pair of nodes.

### Coding Task

In this task, you will code the final attention step used by both GAT and GATv2. Suppose a model has already produced raw scores for a node's neighbors. Your job is to convert those raw scores into attention percentages.

For example, if the current node is **Bob**, and its candidate neighbors are:

| Neighbor | Raw score |
|---|---:|
| Alice | 1.2 |
| Carol | 0.4 |
| Dan | -0.5 |

Write code that applies LeakyReLU, then softmax, then converts the result into percentages that add up to **100%**.


In [ ]:
from math import exp


def leaky_relu(x, negative_slope=0.2):
    # TODO: Return x if it is positive, otherwise return negative_slope * x.
    pass


def softmax(scores):
    # TODO: Convert a list of scores into probabilities that add up to 1.
    pass


def attention_percentages(raw_scores):
    # TODO: Apply leaky_relu to each raw score.
    # TODO: Apply softmax to the activated scores.
    # TODO: Return attention percentages that add up to 100.
    pass


from grader import grade_gat_task

grade_gat_task(leaky_relu, softmax, attention_percentages)


## Task 5: Quantum Data Encoding in QGAT

<div style="background-color: #030303; color: #e6e6e6; padding: 16px; border-left: 4px solid #2edee4; border-radius: 5px;">
<strong>Goal:</strong> Prepare classical node features for a quantum circuit using normalized amplitude encoding.
</div>

Paper reference: https://arxiv.org/pdf/2508.17630

A quantum computer does not directly receive a normal Python list like `[3, 4, 0]`. The list must first be encoded into a quantum state. In the QGAT paper, node features are encoded using **amplitude encoding**: the feature values become the amplitudes of a quantum state.

The important beginner idea is this: amplitudes must be normalized. If the vector is `[3, 4, 0]`, its length is `5`, so the normalized vector becomes `[0.6, 0.8, 0.0]`. Quantum states also need a vector length that is a power of 2, because `n` qubits represent `2^n` amplitudes. So a length-3 vector is padded to length 4 and needs 2 qubits.

### Coding Task

Write two helper functions:

- `qubits_needed(feature_length)`: returns the smallest number of qubits needed.
- `amplitude_encode(features)`: normalizes the vector and pads it with zeros until its length is a power of 2.


In [ ]:
from math import sqrt


def qubits_needed(feature_length):
    # TODO: Return the smallest n such that 2**n >= feature_length.
    pass


def amplitude_encode(features):
    # TODO: Normalize the input features.
    # TODO: Pad the normalized vector with zeros until its length is 2**n.
    # TODO: If all feature values are zero, raise ValueError.
    pass


from grader import grade_quantum_data_encoding

grade_quantum_data_encoding(qubits_needed, amplitude_encode)


## Task 6: Quantum Graph Attention

<div style="background-color: #030303; color: #e6e6e6; padding: 16px; border-left: 4px solid #2edee4; border-radius: 5px;">
<strong>Goal:</strong> Treat quantum measurements as attention logits and normalize them over graph neighbors.
</div>

In classical GAT, attention scores are produced by a classical neural network. In **QGAT**, the paper replaces that scoring part with a variational quantum circuit. For each edge `(i, j)`, the model builds an edge feature, amplitude-encodes it, runs a strongly entangling quantum circuit, and measures Pauli-Z expectation values.

Each measured value acts like a raw attention logit. A useful idea from the paper is that one quantum circuit can produce multiple logits at once, one from each measured qubit. That is how QGAT can behave like multi-head attention while sharing one quantum circuit.

After the quantum circuit gives logits, QGAT normalizes them with softmax over the neighbors of the same node. No extra LeakyReLU is used here, because the quantum circuit itself is already the nonlinear part.

### Coding Task

Suppose the quantum circuit already produced these logits for Bob's candidate neighbors:

| Head | Alice | Carol | Dan |
|---|---:|---:|---:|
| head_1 | 0.2 | 1.1 | -0.4 |
| head_2 | -0.1 | 0.3 | 0.9 |

Write code that converts the logits into attention percentages for each head. Each head should add up to **100%** across Alice, Carol, and Dan.


In [ ]:
from math import exp


def softmax_values(scores):
    # TODO: Convert scores into probabilities that add up to 1.
    pass


def quantum_attention_percentages(quantum_logits):
    # TODO: quantum_logits is a dictionary like:
    # {
    #     "head_1": {"Alice": 0.2, "Carol": 1.1, "Dan": -0.4},
    #     "head_2": {"Alice": -0.1, "Carol": 0.3, "Dan": 0.9},
    # }
    # TODO: Return the same shape, but with attention percentages.
    pass


from grader import grade_qgat_attention

grade_qgat_attention(quantum_attention_percentages)


## Task 7: New QGAT Architecture Design and Benchmarks

<div style="background-color: #030303; color: #e6e6e6; padding: 16px; border-left: 4px solid #2edee4; border-radius: 5px;">
<strong>Goal:</strong> Reconstruct the QGAT layer pipeline and compare its benchmark results against GAT and GATv2.
</div>

The paper's QGAT layer is a hybrid design: classical code prepares graph features, a quantum circuit produces attention logits, and classical graph aggregation updates the node embeddings. The design keeps the familiar GAT workflow, but replaces the attention scorer with an amplitude-encoded, strongly entangling quantum module.

A high-level QGAT layer follows this pipeline:

1. Project node features.
2. Build edge features by combining source and target nodes.
3. Add raw node features as a residual-enhanced input.
4. Compress the vector so it fits amplitude encoding.
5. Amplitude encode the vector into qubits.
6. Run a strongly entangling quantum circuit.
7. Measure Pauli-Z values to get attention logits.
8. Softmax over neighbors.
9. Aggregate neighbor messages across heads.
10. Apply residual connection and dropout.

The paper reports QGAT against GAT and GATv2 on node classification and link prediction benchmarks. Use the table below for the coding task.

| Dataset | GAT | GATv2 | QGAT |
|---|---:|---:|---:|
| Pubmed | 78.10 | 78.50 | 79.20 |
| ogbn-arxiv | 71.54 | 71.87 | 73.62 |
| ogbn-products | 79.04 | 80.63 | 82.10 |
| PPI | 97.30 | 98.20 | 98.90 |
| ogbn-proteins | 78.63 | 79.52 | 79.41 |
| ogbl-collab | 46.63 | 49.70 | 51.20 |
| ogbl-citation2 | 75.95 | 80.14 | 82.20 |

### Coding Task

Fill the architecture steps in order, identify the best model for each dataset, and compute `QGAT - GATv2` for each dataset.


In [ ]:
# TODO: Fill these architecture steps in the correct order.
# Use these exact phrases:
# "project node features"
# "concatenate edge features"
# "add residual raw features"
# "compress for amplitude encoding"
# "amplitude encode"
# "run strongly entangling quantum circuit"
# "measure pauli-z logits"
# "softmax over neighbors"
# "aggregate heads"
# "apply residual and dropout"
architecture_steps = [
    "",
    "",
    "",
    "",
    "",
    "",
    "",
    "",
    "",
    "",
]

# TODO: Fill the model name with the highest score for each dataset.
benchmark_winners = {
    "Pubmed": "",
    "ogbn-arxiv": "",
    "ogbn-products": "",
    "PPI": "",
    "ogbn-proteins": "",
    "ogbl-collab": "",
    "ogbl-citation2": "",
}

# TODO: Fill QGAT score minus GATv2 score for each dataset.
qgat_minus_gatv2 = {
    "Pubmed": 0.0,
    "ogbn-arxiv": 0.0,
    "ogbn-products": 0.0,
    "PPI": 0.0,
    "ogbn-proteins": 0.0,
    "ogbl-collab": 0.0,
    "ogbl-citation2": 0.0,
}

from grader import grade_qgat_architecture_and_benchmarks

grade_qgat_architecture_and_benchmarks(
    architecture_steps,
    benchmark_winners,
    qgat_minus_gatv2,
)


---

## References

1. Vaswani et al., [Attention Is All You Need](https://arxiv.org/abs/1706.03762).
2. An Ning, Tai Yue Li, Nan Yow Chen, [Quantum Graph Attention Network: A Novel Quantum Multi-Head Attention Mechanism for Graph Learning](https://arxiv.org/pdf/2508.17630).
3. Video: https://youtu.be/vkhPtpUiLd8?si=0J3XuVj1JuY1-r5m
4. Video: https://youtu.be/42L1q1Z4Ojc?si=aIQzBxork-3tLBki
